In [1]:
import json
import time
from pathlib import Path
from collections import Counter
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import urllib3

urllib3.disable_warnings()

BASE = "https://115.159.159.82"
print("环境就绪")

环境就绪


In [2]:
api_url = BASE + "/api/devices"

r = requests.get(
    api_url,
    params={"page": 1, "page_size": 50},
    timeout=10,
    verify=False
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [3]:
data = r.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [4]:
print(data)

{'page': 1, 'page_size': 50, 'total': 5000, 'items': [{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}, {'device_id': 'D00000002', 'company_id': 'C000001', 'device_name': '循环泵-000001-002', 'device_type': 'pump', 'model': 'SP-PUM-500', 'zone': '智造一区', 'install_date': '2016-03-06', 'rated_power_kw': 73.8, 'sampling_interval_s': 60, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-08T05:44:00'}, {'device_id': 'D00000003', 'company_id': 'C000001', 'device_name': '空压机-000001-003', 'device_type': 'compressor', 'model': 'SP-COM-200', 'zone': '智造一区', 'install_date': '2016-12-18', 'rated_power_kw': 248.6, 'sampling_interval_s': 30, 'protocol': 'modbus', 'status': 'running', 'updated_at': '2026-08-29T04:01:00'}, {'device_id': 'D

In [5]:
print(data["items"][0])

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [6]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [7]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
            "page": page,
            "page_size": page_size
        },
        timeout=10,
        verify=False
    )

    page_data = r.json()

    devices.extend(page_data["items"])

print("device count:", len(devices))

device count: 5000


In [8]:
from bs4 import BeautifulSoup

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10, verify=False)
soup = BeautifulSoup(r.text, "html.parser")

In [9]:
cards = soup.select("article.company-card")

print("本页企业数：", len(cards))

本页企业数： 20


In [10]:
card = cards[0]

details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [11]:
def parse_company(card):
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    return {
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "status": card.select_one(".badge").get_text(strip=True),
        "industry": details.get("所属行业"),
        "zone": details.get("所在区域"),
        "device_count": int(details.get("设备数量", "0").replace("台", "").strip()),
    }


print(parse_company(cards[0]))

{'company_id': 'C000001', 'name': '临港精工001有限公司', 'status': '运营中', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6}


In [12]:
companies = []

for card in cards:
    companies.append(parse_company(card))

print("本页企业数：", len(companies))
print(companies[0])

本页企业数： 20
{'company_id': 'C000001', 'name': '临港精工001有限公司', 'status': '运营中', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6}


In [13]:
next_link = soup.select_one("nav.pagination a:last-of-type")

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [14]:
companies = []
current_url = BASE + "/companies"
page_no = 0

while page_no < 3:
    page_no += 1

    r = requests.get(current_url, timeout=10, verify=False)
    soup = BeautifulSoup(r.text, "html.parser")

    cards = soup.select("article.company-card")
    page_companies = [parse_company(c) for c in cards]
    companies.extend(page_companies)

    print("第", page_no, "页：", len(page_companies), "条")

    next_link = None
    for a in soup.select("nav.pagination a"):
        if a.get_text(strip=True) == "下一页":
            next_link = a
            break

    if next_link is None:
        print("没有下一页了")
        break

    current_url = urljoin(current_url, next_link.get("href"))
    print("   下一页 ->", current_url)

print("Company 总数：", len(companies))

第 1 页： 20 条
   下一页 -> https://115.159.159.82/companies?page=2
第 2 页： 20 条
   下一页 -> https://115.159.159.82/companies?page=3
第 3 页： 20 条
   下一页 -> https://115.159.159.82/companies?page=4
Company 总数： 60


In [15]:
print("companies:", len(companies))
print(
    "unique company_id:",
    len(set(x["company_id"] for x in companies))
)

print("devices:", len(devices))
print(
    "unique device_id:",
    len(set(x["device_id"] for x in devices))
)

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000


In [16]:
from collections import Counter

device_counts = Counter(
    x["company_id"] for x in devices
)

mismatch_count = 0

for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)

    if expected != actual:
        mismatch_count += 1
        print("不一致:", cid, "页面写", expected, "实际", actual)

print("Company-Device 不一致企业数:", mismatch_count)

Company-Device 不一致企业数: 0


In [17]:
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录


In [18]:
print("devices.json  :", (data_dir / "devices.json").stat().st_size, "字节")
print("companies.json:", (data_dir / "companies.json").stat().st_size, "字节")

with open(data_dir / "devices.json", encoding="utf-8") as f:
    check = json.load(f)

print("读回来 devices:", len(check), "条")

devices.json  : 1907611 字节
companies.json: 11520 字节
读回来 devices: 5000 条
